# 05.01 - GenAIServer: serving GenAI models over HTTP

**Why.** In `02-run-llm-vlm` we called models *in process* with
`pyneat.genai.GenAIModel(...).run(request)`. That is the right default. But sometimes the natural
application boundary is **HTTP**: a browser UI, a companion service, or a remote client that should
not link against the Neat runtime. For that, Neat gives you `pyneat.genai.GenAIServer` - an
in-process HTTP server that exposes **OpenAI-compatible** endpoints and can host several models at
once.

This notebook explains the concepts and shows the exact commands. Starting a server loads models onto
the MLA, so every start/request is **HEAVY / MANUAL RUN - not executed by tooling** and is shown as a
printed string. Grounded in `/workspace/core/tutorials/021_serve_genai_models` and the headers under
`/workspace/core/include/genai/`.

## Do NOT confuse the two "servers"

There are two different things with "server" in the name. Conflating them will mislead you.

| | `llima benchmark-server` (CLI) | `pyneat.genai.GenAIServer` (API) |
| --- | --- | --- |
| What | on-board CLI subcommand for **benchmarking one model** | in-process Python/C++ **application server** |
| Where | `/usr/bin/llima` on the DevKit | your own Python process (`import pyneat`) |
| Models | one (`benchmark-server [--port P] <model>`) | many, each with a served name |
| Purpose | measure a single model's throughput | serve an app's LLM/VLM/ASR endpoints |
| Source | `<scratchpad>/llima_ground_truth.md` | `core/include/genai/GenAIServer.h`, tutorial 021 |

This section (05) is about **`GenAIServer`**, the API. The `llima benchmark-server` CLI is a separate
quick-benchmark tool covered in `01-llima-basics`. (Reminder trap: the `llima` serve subcommand is
`benchmark-server`, **not** `serve`.)

## Direct model vs server - when to use which

| | Direct `GenAIModel` / `VisionLanguageModel` / `ASRModel` | `GenAIServer` |
| --- | --- | --- |
| Boundary | in-process function call | HTTP over the network |
| Overhead | lowest | HTTP + serialization |
| Best for | embedded app logic, tests, tight loops | UIs, remote clients, multi-language clients |
| Multi-model | you manage each handle | one process, many served names |
| Clients | must link/import pyneat | any HTTP client (curl, `requests`, JS) |

Rule from the tutorial 021 "In Practice": *use direct calls for lower-overhead application code inside
the same process; use the server when a network boundary is useful.*

## The GenAIServer API surface [core]

Verified against `/workspace/core/include/genai/GenAIServer.h` and the Python bindings in
`/workspace/core/python/src/module.cpp` (~line 2216):

- `GenAIServerOptions` - fields `host` (default `"0.0.0.0"`) and `port` (default `9998`).
- `GenAIServer(options)` - construct.
- `add_model(model_dir)` -> auto-derived served name; or `add_model(model_dir, served_name)`.
- `remove_model(served_name)` -> bool.
- `model_names()` -> list of served names.
- `serve()` - **blocking** foreground loop.
- `start()` / `stop()` - **non-blocking**; use when your app owns the process lifetime.

The served name is what the client puts in the request's `model` field.

In [ ]:
# HEAVY / MANUAL RUN - not executed by tooling. Constructing the server and
# calling add_model()/start() LOADS models onto the MLA. Shown as source text so
# running this cell does nothing but print. Owner runs the real thing on the board.

server_snippet = """
import pyneat as neat

options = neat.genai.GenAIServerOptions()
options.host = "0.0.0.0"   # reachable from other machines on the network
options.port = 9998        # default

server = neat.genai.GenAIServer(options)
server.add_model("/media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4", "llm")
print("registered:", server.model_names())   # -> ['llm']

server.start()   # non-blocking; serve() would block instead
# ... application runs ...
# server.stop()
"""
print("HEAVY - documented, not executed. Server setup:")
print(server_snippet)

## OpenAI-compatible endpoints

Once the server is up, clients speak a subset of the OpenAI HTTP API (from tutorial 021):

| Endpoint | Method | Used by |
| --- | --- | --- |
| `/v1/models` | GET | list served names - the quickest smoke check |
| `/v1/chat/completions` | POST | LLM text chat and VLM image+text chat |
| `/v1/audio/transcriptions` | POST (multipart) | ASR transcription |

Requests set `"stream": true` to receive Server-Sent Events; the tutorial clients also surface
server-side `ttft` (time to first token) and per-token `tps`.

### Launch the server (documented commands, owner-run)

Uses the tutorial 021 script, or our `scripts/serve_multi_model.py`. Printed strings only.

In [ ]:
# HEAVY / MANUAL RUN - not executed by tooling. Commands for the owner to run on the board.

# Human, real terminal (intended UX):
dk_launch = ("dk /workspace/demo-neat/llima/05-genai-server/scripts/serve_multi_model.py "
             "--llm /media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4 "
             "--vlm '' --asr ''")

# Automation / CI (no TTY) fallback:
ssh_launch = ("timeout 600 ssh -o BatchMode=yes sima@192.168.135.203 "
              "'source $HOME/pyneat/bin/activate; "
              "python /workspace/demo-neat/llima/05-genai-server/scripts/serve_multi_model.py "
              "--vlm \"\" --asr \"\"'")

for label, cmd in [("dk (human)", dk_launch), ("ssh (CI)", ssh_launch)]:
    print(f"[{label}]"); print(" ", cmd); print()

### Smoke-test and call the endpoints (documented, owner-run)

`/v1/models` first, then a real request. These hit a **running** server; they do not load a model.

In [ ]:
# MANUAL RUN - talks to a running server. Printed strings only.

MODALIX_IP = "192.168.135.203"  # replace with your board IP

smoke = f"curl http://{MODALIX_IP}:9998/v1/models"
# expected: JSON listing the served names, e.g. {"data":[{"id":"llm",...}]}

text_chat = (
    f"curl http://{MODALIX_IP}:9998/v1/chat/completions "
    f"-H 'Content-Type: application/json' -d '"
    '{"model":"llm","messages":[{"role":"user","content":"Explain an API gateway in one sentence."}],'
    '"stream":true}' "'"
)

# Or use our client (see scripts/client_examples.py, adapted from tutorial 021):
client_text = ("python /workspace/demo-neat/llima/05-genai-server/scripts/client_examples.py "
               f"--run text --server-ip {MODALIX_IP} 'Explain an API gateway in one sentence.'")

for label, cmd in [("GET /v1/models", smoke), ("curl chat", text_chat), ("client_examples", client_text)]:
    print(f"[{label}]"); print(" ", cmd); print()

## Expected output

- `GET /v1/models` returns JSON whose `data[].id` values are the served names you registered
  (e.g. `llm`). This is the "is the server alive and populated?" check from tutorial 021.
- A streamed `/v1/chat/completions` request prints the answer token-by-token, then reports server
  `ttft` and average/min/max `tps` (the tutorial clients parse these from the SSE events).
- `server.model_names()` in-process returns the same served names.

If `/v1/models` is unreachable, the server did not start or the port is blocked. If it returns an
empty list, no `add_model(...)` succeeded - check the model directory paths against
`/media/nvme/llima/models/`.